# Lab 1 — Prepare robot data for OpenVLA

This lab fine-tunes **OpenVLA-7B** (a vision-language-action model: SigLIP + DINOv2 vision, Llama-2 7B
language backbone, actions emitted as 7 discrete tokens) with **LoRA** on a **SageMaker Training Job**.
It follows the workflow of the *Serverless Model Customization* SFT lab (prepare data -> train -> evaluate ->
register -> deploy), but VLAs are not available in serverless customization, so training runs as a
`ModelTrainer` job with our own script, as in the AWS blog
[Fine-tuning OpenVLA on Amazon SageMaker AI with LoRA](https://aws.amazon.com/blogs/physical-ai/fine-tuning-openvla-on-amazon-sagemaker-ai-with-lora/).

In this notebook you build the training data. `config.DATASET` picks it:

* **`"libero"` (default):** demonstrations from [LIBERO](https://libero-project.github.io/), a simulated tabletop
  benchmark with 40 tasks ("turn on the stove and put the moka pot on it", ...) and real 7-DoF end-effector
  actions, from [physical-intelligence/libero](https://huggingface.co/datasets/physical-intelligence/libero)
  (CC BY 4.0). OpenVLA was not pretrained on LIBERO; the OpenVLA paper uses it to show what fine-tuning adds.
* **`"bridge"`:** [BridgeData V2 (scripted, images)](https://huggingface.co/datasets/VyoJ/BridgeData-V2-Scripted-Images)
  with x/y actions pseudo-labelled from optical flow and one instruction, as in the AWS blog. Useful to compare,
  but the model mostly learns the label prior.

Both produce the same format: an image, an instruction and a 7-DoF action normalized to [-1, 1] the way OpenVLA
normalizes its training data (per-axis 1st/99th percentile bounds), uploaded to S3 as a Hugging Face `DatasetDict`.

> **Bring your own data:** produce the same columns (image, instruction, 7-DoF action, episode id) from your robot
> logs. Everything downstream stays the same.


In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

import config

sess = Session()
try:
    role = get_execution_role()
except ValueError:
    role = boto3.client("iam").get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

region = sess.boto_region_name
bucket_name = sess.default_bucket()
s3_root = config.s3_root(bucket_name, sess.default_bucket_prefix)
dataset_s3_uri = f"{s3_root}/datasets/{config.DATASET_PREFIX}/"
s3 = boto3.client("s3", region_name=region)
sm_client = boto3.client("sagemaker", region_name=region)

print(f"role:     {role}")
print(f"region:   {region}")
print(f"run mode: {config.RUN_MODE}, capacity: {config.CAPACITY} -> {', '.join(config.TRAIN['instance_types'])}")
print(f"s3 root:  {s3_root}")

## Build the dataset

Validation holds out whole episodes, so the model is scored on demonstrations it has not seen (for LIBERO, one
per task). The default source is a sample shipped with the lab, so this notebook needs no Hugging Face access:

* `sample_data/libero-120`: 120 LIBERO demonstrations (3 per task), every 5th frame, 224x224 JPEG.
* `sample_data/bridge-300`: 300 Bridge episodes (images downscaled to 320x240).

Set `SOURCE_PARQUET_DIR = None` to download from the Hub instead (log in with `huggingface_hub.login()` first if
you get HTTP 429).


In [ ]:
import json
from tqdm.auto import tqdm

import importlib

build_dataset = importlib.import_module(f"{config.DATASET}_data").build_dataset   # libero_data or bridge_data

# Data source for this run (defaults come from config.py):
#   config default           -> the sample shipped with the lab for config.DATASET (no Hugging Face access needed)
#   None                     -> download from the Hugging Face Hub (log in with a token if you get HTTP 429)
#   "<folder>"               -> Parquet shards you downloaded yourself
SOURCE_PARQUET_DIR = config.SOURCE_PARQUET_DIR
MAX_EPISODES = config.MAX_EPISODES   # bridge only

print(f"dataset: {config.DATASET}, source: {SOURCE_PARQUET_DIR or 'Hugging Face Hub'}")
dataset, action_stats = build_dataset(max_episodes=MAX_EPISODES, val_fraction=config.VAL_FRACTION,
                                      progress=tqdm, parquet_dir=SOURCE_PARQUET_DIR)
dataset

In [ ]:
import numpy as np

train_actions = np.array(dataset["train"]["action"])
# LIBERO: the recorded 7-DoF actions; bridge: pixels of mean optical flow (x/y)
print("normalization bounds (q01 / q99 of the training actions):")
for axis, lo, hi in zip(["x", "y", "z", "roll", "pitch", "yaw", "gripper"], action_stats["q01"], action_stats["q99"]):
    print(f"  {axis}: q01={lo:+.3f}  q99={hi:+.3f}")
print("\nnormalized train actions, mean |a| per axis:", np.abs(train_actions).mean(0).round(3))
print("instruction:", dataset["train"][0]["instruction"])
dataset["train"][0]["image"]

## Upload to S3

`save_to_disk` writes Arrow files that the training script reads with `load_from_disk`.
`action_stats.json` travels with the data, then with the model, so the endpoint can return un-normalized actions.

In [ ]:
from pathlib import Path

local_dir = Path("data") / config.DATASET_PREFIX
dataset.save_to_disk(str(local_dir))
(local_dir / "action_stats.json").write_text(json.dumps(action_stats, indent=2))

bucket, prefix = dataset_s3_uri[5:].split("/", 1)
files = [f for f in local_dir.rglob("*") if f.is_file()]
for f in tqdm(files, desc="upload"):
    s3.upload_file(str(f), bucket, prefix + str(f.relative_to(local_dir)))
print(f"dataset: {dataset_s3_uri}  ({len(files)} files)")

Continue with **2-fine-tune-vla.ipynb**.